In [1]:
#!/usr/bin/env python3
# Modified script with enhanced visualizations for HAL, KDE, and Kaplan-Meier comparisons

# Import needed libraries
import numpy as np
from numpy import trapezoid  # Use trapezoid instead of trapz
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import truncnorm, ttest_rel, linregress
from sklearn.neighbors import KernelDensity
from lifelines import KaplanMeierFitter
import time
from matplotlib.gridspec import GridSpec

# Set global parameters
n_experiments = 500  # Total number of experiments in HAL results
n_generate = 500  # Number of experiments to generate for each sample size
smoothness_order = 1
ridge = 1e-6
sample_sizes = [200, 400, 800, 1600, 3200]  # All sample sizes to analyze

# Directory structure adjustment
hal_dir = "../Asymptoticity_FD_0301/out"  # HAL results directory
comparison_base_dir = "."  # Current directory
evaluation_points = np.linspace(0.02, 0.98, 20)

# Set style for plots
sns.set_style("whitegrid")
plt.rcParams.update({'font.size': 12})

# Create containers for cross-sample size summary metrics
sample_size_summary = {
    'sample_size': [],
    # Survival function metrics
    'hal_init_surv_mean_abs_bias': [], 'hal_final_surv_mean_abs_bias': [], 
    'kde_surv_mean_abs_bias': [], 'km_surv_mean_abs_bias': [],
    'hal_init_surv_mean_variance': [], 'hal_final_surv_mean_variance': [], 
    'kde_surv_mean_variance': [], 'km_surv_mean_variance': [],
    'hal_init_surv_mean_mse': [], 'hal_final_surv_mean_mse': [], 
    'kde_surv_mean_mse': [], 'km_surv_mean_mse': [],
    # Density function metrics
    'hal_init_dens_mean_abs_bias': [], 'hal_final_dens_mean_abs_bias': [], 'kde_dens_mean_abs_bias': [],
    'hal_init_dens_mean_variance': [], 'hal_final_dens_mean_variance': [], 'kde_dens_mean_variance': [],
    'hal_init_dens_mean_mse': [], 'hal_final_dens_mean_mse': [], 'kde_dens_mean_mse': []
}

# Function to generate truncated normal data
def generate_truncated_normal(n_samples, mean=0.5, std=0.1, lower=0, upper=1, seed=None):
    if seed is not None:
        np.random.seed(seed)
    a, b = (lower - mean) / std, (upper - mean) / std
    T = truncnorm.rvs(a, b, loc=mean, scale=std, size=n_samples)
    return T

# KDE with bandwidth selection (updated to use trapezoid)
def estimate_density_kde(data, evaluation_points, bandwidth=None):
    """
    Estimate density using KDE with automatic bandwidth selection if not provided.
    """
    if bandwidth is None:
        # Scott's rule for bandwidth selection
        bandwidth = 1.06 * np.std(data) * len(data)**(-1/5)
    
    # Fit KDE
    kde = KernelDensity(bandwidth=bandwidth, kernel='gaussian')
    kde.fit(data.reshape(-1, 1))
    
    # Get log density and convert to density
    log_density = kde.score_samples(evaluation_points.reshape(-1, 1))
    density = np.exp(log_density)
    
    # Normalize to ensure it integrates to 1 over [0,1]
    density = density / trapezoid(density, evaluation_points)
    
    return density, kde

# Compute survival function from KDE
def compute_survival_from_kde(kde, evaluation_points):
    """
    Compute survival function from KDE by numerical integration.
    """
    survival = np.zeros_like(evaluation_points)
    grid = np.linspace(0, 1, 1000)  # Fine grid for integration
    
    log_density_grid = kde.score_samples(grid.reshape(-1, 1))
    density_grid = np.exp(log_density_grid)
    density_grid = density_grid / trapezoid(density_grid, grid)  # Normalize
    
    for i, point in enumerate(evaluation_points):
        # S(x) = integral of f(t) dt from x to 1
        mask = grid >= point
        if any(mask):
            survival[i] = trapezoid(density_grid[mask], grid[mask])
        else:
            survival[i] = 0
    
    return survival

# Kaplan-Meier estimator
def estimate_survival_km(data, evaluation_points):
    """
    Estimate survival function using Kaplan-Meier.
    """
    kmf = KaplanMeierFitter()
    kmf.fit(data)
    survival = kmf.survival_function_at_times(evaluation_points).values
    return survival.flatten()

# Function to compute true density and survival
def compute_true_functions(evaluation_points, mean=0.5, std=0.1, lower=0, upper=1):
    """
    Compute true density and survival for truncated normal.
    """
    a, b = (lower - mean) / std, (upper - mean) / std
    true_density = truncnorm.pdf(evaluation_points, a, b, loc=mean, scale=std)
    true_survival = 1 - truncnorm.cdf(evaluation_points, a, b, loc=mean, scale=std)
    return true_density, true_survival

# Modified load_data function that uses separate directories
def load_data_fixed(hal_dir, comp_dir, n_samples, smoothness_order, ridge):
    """
    Modified load_data function that uses separate directories for HAL and comparison data.
    """
    # Load HAL results
    hal_base = f"{hal_dir}/hal_n{n_samples}_smooth{smoothness_order}_ridge{ridge}"
    hal_data = {
        "initial_density": pd.read_csv(f"{hal_base}_initial_density.csv", index_col=0),
        "final_density": pd.read_csv(f"{hal_base}_final_density.csv", index_col=0),
        "initial_survival": pd.read_csv(f"{hal_base}_initial_survival.csv", index_col=0),
        "final_survival": pd.read_csv(f"{hal_base}_final_survival.csv", index_col=0),
        "initial_density_se": pd.read_csv(f"{hal_base}_initial_density_se.csv", index_col=0),
        "initial_survival_se": pd.read_csv(f"{hal_base}_initial_survival_se.csv", index_col=0),
        "initial_survival_se_eic": pd.read_csv(f"{hal_base}_initial_survival_se_eic.csv", index_col=0),
        "final_survival_se": pd.read_csv(f"{hal_base}_final_survival_se.csv", index_col=0),
        "final_survival_se_eic": pd.read_csv(f"{hal_base}_final_survival_se_eic.csv", index_col=0)
    }
    
    # Load comparison results from separate directory
    comp_base = f"{comp_dir}/comparison_n{n_samples}"
    comparison_data = {
        "kde_density": pd.read_csv(f"{comp_base}_kde_density.csv", index_col=0),
        "kde_survival": pd.read_csv(f"{comp_base}_kde_survival.csv", index_col=0),
        "km_survival": pd.read_csv(f"{comp_base}_km_survival.csv", index_col=0),
        "true_density": pd.read_csv(f"{comp_base}_true_density.csv", index_col=0),
        "true_survival": pd.read_csv(f"{comp_base}_true_survival.csv", index_col=0)
    }
    
    return hal_data, comparison_data

# Function to generate comparison data for a specific sample size
def generate_comparison_data(n_samples, n_generate, evaluation_points, out_comparison_dir):
    """Generate comparison data for a specific sample size"""
    print(f"Generating comparison data for {n_generate} experiments with n={n_samples} samples each...")
    
    # Initialize DataFrames to store results - use n_experiments to match HAL data
    exp_ids = [f"Exp_{i+1}" for i in range(n_experiments)]
    
    # Initialize with NaN to indicate missing data
    kde_density_df = pd.DataFrame(index=evaluation_points, columns=exp_ids)
    kde_survival_df = pd.DataFrame(index=evaluation_points, columns=exp_ids)
    km_survival_df = pd.DataFrame(index=evaluation_points, columns=exp_ids)
    true_density_df = pd.DataFrame(index=evaluation_points, columns=exp_ids)
    true_survival_df = pd.DataFrame(index=evaluation_points, columns=exp_ids)
    
    # Run experiments sequentially
    for i in range(n_generate):
        if i % 10 == 0:
            print(f"Processing experiment {i+1}/{n_generate}...")
        
        # Set seed
        seed = 12776 + i
        
        # Generate data
        T = generate_truncated_normal(n_samples, seed=seed)
        
        # KDE for density and survival
        kde_density, kde_model = estimate_density_kde(T, evaluation_points)
        kde_survival = compute_survival_from_kde(kde_model, evaluation_points)
        
        # Kaplan-Meier for survival
        km_survival = estimate_survival_km(T, evaluation_points)
        
        # True density and survival for reference
        true_density = truncnorm.pdf(evaluation_points, 
                                    (0-0.5)/0.1, (1-0.5)/0.1, 
                                    loc=0.5, scale=0.1)
        true_density = true_density / trapezoid(true_density, evaluation_points)
        
        true_survival = 1 - truncnorm.cdf(evaluation_points, 
                                        (0-0.5)/0.1, (1-0.5)/0.1, 
                                        loc=0.5, scale=0.1)
        
        # Store results for this experiment
        kde_density_df[f"Exp_{i+1}"] = kde_density
        kde_survival_df[f"Exp_{i+1}"] = kde_survival
        km_survival_df[f"Exp_{i+1}"] = km_survival
        true_density_df[f"Exp_{i+1}"] = true_density
        true_survival_df[f"Exp_{i+1}"] = true_survival
    
    # Save results
    print(f"Saving comparison results to {out_comparison_dir}...")
    
    kde_density_df.to_csv(f"{out_comparison_dir}/comparison_n{n_samples}_kde_density.csv", index=True)
    kde_survival_df.to_csv(f"{out_comparison_dir}/comparison_n{n_samples}_kde_survival.csv", index=True)
    km_survival_df.to_csv(f"{out_comparison_dir}/comparison_n{n_samples}_km_survival.csv", index=True)
    true_density_df.to_csv(f"{out_comparison_dir}/comparison_n{n_samples}_true_density.csv", index=True)
    true_survival_df.to_csv(f"{out_comparison_dir}/comparison_n{n_samples}_true_survival.csv", index=True)
    
    return (kde_density_df, kde_survival_df, km_survival_df, true_density_df, true_survival_df)

# Function to compute performance metrics (bias, variance, MSE) across evaluation points
def compute_performance_metrics(hal_data, comparison_data):
    """
    Compute performance metrics (bias, variance, MSE) for all methods across evaluation points.
    Returns a dictionary with all metrics for both density and survival.
    """
    # Extract evaluation points and common experiment IDs
    eval_points = hal_data["initial_survival"].index.astype(float).values
    hal_exps = set(hal_data["initial_survival"].columns)
    comp_exps = set(comparison_data["km_survival"].columns)
    common_exps = sorted(list(hal_exps.intersection(comp_exps)))
    
    if len(common_exps) == 0:
        print("No common experiments found. Cannot compute metrics.")
        return None
    
    print(f"Computing metrics using {len(common_exps)} common experiments.")
    
    # Get true density and survival functions
    true_density, true_survival = compute_true_functions(eval_points)
    
    # Initialize arrays for metrics
    n_points = len(eval_points)
    
    # ----- SURVIVAL FUNCTION METRICS -----
    # Bias calculation for survival
    hal_init_surv_bias = np.zeros(n_points)
    hal_final_surv_bias = np.zeros(n_points)
    kde_surv_bias = np.zeros(n_points)
    km_surv_bias = np.zeros(n_points)
    
    # Variance calculation for survival
    hal_init_surv_variance = np.zeros(n_points)
    hal_final_surv_variance = np.zeros(n_points)
    kde_surv_variance = np.zeros(n_points)
    km_surv_variance = np.zeros(n_points)
    
    # MSE calculation for survival
    hal_init_surv_mse = np.zeros(n_points)
    hal_final_surv_mse = np.zeros(n_points)
    kde_surv_mse = np.zeros(n_points)
    km_surv_mse = np.zeros(n_points)
    
    # Extract data for common experiments - Survival
    hal_init_surv_data = hal_data["initial_survival"][common_exps].values
    hal_final_surv_data = hal_data["final_survival"][common_exps].values
    kde_surv_data = comparison_data["kde_survival"][common_exps].values
    km_surv_data = comparison_data["km_survival"][common_exps].values
    
    # ----- DENSITY FUNCTION METRICS -----
    # Bias calculation for density
    hal_init_dens_bias = np.zeros(n_points)
    hal_final_dens_bias = np.zeros(n_points)
    kde_dens_bias = np.zeros(n_points)
    
    # Variance calculation for density
    hal_init_dens_variance = np.zeros(n_points)
    hal_final_dens_variance = np.zeros(n_points)
    kde_dens_variance = np.zeros(n_points)
    
    # MSE calculation for density
    hal_init_dens_mse = np.zeros(n_points)
    hal_final_dens_mse = np.zeros(n_points)
    kde_dens_mse = np.zeros(n_points)
    
    # Extract data for common experiments - Density
    hal_init_dens_data = hal_data["initial_density"][common_exps].values
    hal_final_dens_data = hal_data["final_density"][common_exps].values
    kde_dens_data = comparison_data["kde_density"][common_exps].values
    
    # Compute metrics for survival function
    for i in range(n_points):
        # Extract survival estimates at this evaluation point
        hal_init_surv_estimates = hal_init_surv_data[i, :]
        hal_final_surv_estimates = hal_final_surv_data[i, :]
        kde_surv_estimates = kde_surv_data[i, :]
        km_surv_estimates = km_surv_data[i, :]
        
        # True survival value at this point
        true_surv_value = true_survival[i]
        
        # Compute bias (mean estimate - true value) - Survival
        hal_init_surv_bias[i] = np.mean(hal_init_surv_estimates) - true_surv_value
        hal_final_surv_bias[i] = np.mean(hal_final_surv_estimates) - true_surv_value
        kde_surv_bias[i] = np.mean(kde_surv_estimates) - true_surv_value
        km_surv_bias[i] = np.mean(km_surv_estimates) - true_surv_value
        
        # Compute variance - Survival
        hal_init_surv_variance[i] = np.var(hal_init_surv_estimates)
        hal_final_surv_variance[i] = np.var(hal_final_surv_estimates)
        kde_surv_variance[i] = np.var(kde_surv_estimates)
        km_surv_variance[i] = np.var(km_surv_estimates)
        
        # Compute MSE (mean squared error) - Survival
        hal_init_surv_mse[i] = np.mean((hal_init_surv_estimates - true_surv_value) ** 2)
        hal_final_surv_mse[i] = np.mean((hal_final_surv_estimates - true_surv_value) ** 2)
        kde_surv_mse[i] = np.mean((kde_surv_estimates - true_surv_value) ** 2)
        km_surv_mse[i] = np.mean((km_surv_estimates - true_surv_value) ** 2)
        
        # Extract density estimates at this evaluation point
        hal_init_dens_estimates = hal_init_dens_data[i, :]
        hal_final_dens_estimates = hal_final_dens_data[i, :]
        kde_dens_estimates = kde_dens_data[i, :]
        
        # True density value at this point
        true_dens_value = true_density[i]
        
        # Compute bias (mean estimate - true value) - Density
        hal_init_dens_bias[i] = np.mean(hal_init_dens_estimates) - true_dens_value
        hal_final_dens_bias[i] = np.mean(hal_final_dens_estimates) - true_dens_value
        kde_dens_bias[i] = np.mean(kde_dens_estimates) - true_dens_value
        
        # Compute variance - Density
        hal_init_dens_variance[i] = np.var(hal_init_dens_estimates)
        hal_final_dens_variance[i] = np.var(hal_final_dens_estimates)
        kde_dens_variance[i] = np.var(kde_dens_estimates)
        
        # Compute MSE (mean squared error) - Density
        hal_init_dens_mse[i] = np.mean((hal_init_dens_estimates - true_dens_value) ** 2)
        hal_final_dens_mse[i] = np.mean((hal_final_dens_estimates - true_dens_value) ** 2)
        kde_dens_mse[i] = np.mean((kde_dens_estimates - true_dens_value) ** 2)
    
    # Return all metrics
    metrics = {
        'eval_points': eval_points,
        # Survival metrics
        'hal_init_surv_bias': hal_init_surv_bias,
        'hal_final_surv_bias': hal_final_surv_bias,
        'kde_surv_bias': kde_surv_bias,
        'km_surv_bias': km_surv_bias,
        'hal_init_surv_variance': hal_init_surv_variance,
        'hal_final_surv_variance': hal_final_surv_variance,
        'kde_surv_variance': kde_surv_variance,
        'km_surv_variance': km_surv_variance,
        'hal_init_surv_mse': hal_init_surv_mse,
        'hal_final_surv_mse': hal_final_surv_mse,
        'kde_surv_mse': kde_surv_mse,
        'km_surv_mse': km_surv_mse,
        # Density metrics
        'hal_init_dens_bias': hal_init_dens_bias,
        'hal_final_dens_bias': hal_final_dens_bias,
        'kde_dens_bias': kde_dens_bias,
        'hal_init_dens_variance': hal_init_dens_variance,
        'hal_final_dens_variance': hal_final_dens_variance,
        'kde_dens_variance': kde_dens_variance,
        'hal_init_dens_mse': hal_init_dens_mse,
        'hal_final_dens_mse': hal_final_dens_mse,
        'kde_dens_mse': kde_dens_mse
    }
    
    return metrics

# New function to plot side-by-side bias, variance, and MSE for a single sample size
def plot_metrics_by_evaluation_point(metrics, output_dir, n_samples):
    """
    Create side-by-side plots of bias, variance, and MSE across evaluation points
    for both density and survival functions.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Get evaluation points
    eval_points = metrics['eval_points']
    
    # ----- SURVIVAL FUNCTION PLOTS -----
    # Create figure with 3 subplots side by side for survival function
    fig_surv = plt.figure(figsize=(18, 6))
    gs_surv = GridSpec(1, 3, figure=fig_surv, wspace=0.3)
    
    # Plot 1: Bias across evaluation points for survival
    ax1_surv = fig_surv.add_subplot(gs_surv[0, 0])
    ax1_surv.plot(eval_points, metrics['km_surv_bias'], 'bo-', label='KM Bias', linewidth=2, markersize=6)
    ax1_surv.plot(eval_points, metrics['hal_init_surv_bias'], 'orange', marker='o', linestyle='-', 
             label='Initial Bias', linewidth=2, markersize=6)
    ax1_surv.plot(eval_points, metrics['hal_final_surv_bias'], 'go-', label='Final Bias', linewidth=2, markersize=6)
    ax1_surv.plot(eval_points, metrics['kde_surv_bias'], 'ro-', label='KDE Bias', linewidth=2, markersize=6)
    
    ax1_surv.set_xlabel('Evaluation Point')
    ax1_surv.set_ylabel('Bias')
    ax1_surv.set_title('Bias Across Evaluation Points')
    ax1_surv.grid(True)
    ax1_surv.legend()
    
    # Plot 2: Variance across evaluation points for survival
    ax2_surv = fig_surv.add_subplot(gs_surv[0, 1])
    ax2_surv.plot(eval_points, metrics['km_surv_variance'], 'bo-', label='KM Variance', linewidth=2, markersize=6)
    ax2_surv.plot(eval_points, metrics['hal_init_surv_variance'], 'orange', marker='o', linestyle='-',
             label='Initial Variance', linewidth=2, markersize=6)
    ax2_surv.plot(eval_points, metrics['hal_final_surv_variance'], 'go-', label='Final Variance', linewidth=2, markersize=6)
    ax2_surv.plot(eval_points, metrics['kde_surv_variance'], 'ro-', label='KDE Variance', linewidth=2, markersize=6)
    
    ax2_surv.set_xlabel('Evaluation Point')
    ax2_surv.set_ylabel('Variance')
    ax2_surv.set_title('Variance Across Evaluation Points')
    ax2_surv.grid(True)
    ax2_surv.legend()
    
    # Plot 3: MSE across evaluation points for survival
    ax3_surv = fig_surv.add_subplot(gs_surv[0, 2])
    ax3_surv.plot(eval_points, metrics['km_surv_mse'], 'bo-', label='KM MSE', linewidth=2, markersize=6)
    ax3_surv.plot(eval_points, metrics['hal_init_surv_mse'], 'orange', marker='o', linestyle='-',
             label='Initial MSE', linewidth=2, markersize=6)
    ax3_surv.plot(eval_points, metrics['hal_final_surv_mse'], 'go-', label='Final MSE', linewidth=2, markersize=6)
    ax3_surv.plot(eval_points, metrics['kde_surv_mse'], 'ro-', label='KDE MSE', linewidth=2, markersize=6)
    
    ax3_surv.set_xlabel('Evaluation Point')
    ax3_surv.set_ylabel('MSE')
    ax3_surv.set_title('MSE Across Evaluation Points')
    ax3_surv.grid(True)
    ax3_surv.legend()
    
    # Add overall title for survival plots
    plt.suptitle(f'Survival Function Metrics for Sample Size n={n_samples}', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for the suptitle
    
    # Save the figure for survival
    fig_surv.savefig(f"{output_dir}/survival_metrics_n{n_samples}.png", dpi=300, bbox_inches='tight')
    plt.close(fig_surv)
    
    # ----- DENSITY FUNCTION PLOTS -----
    # Create figure with 3 subplots side by side for density function
    fig_dens = plt.figure(figsize=(18, 6))
    gs_dens = GridSpec(1, 3, figure=fig_dens, wspace=0.3)
    
    # Plot 1: Bias across evaluation points for density
    ax1_dens = fig_dens.add_subplot(gs_dens[0, 0])
    ax1_dens.plot(eval_points, metrics['hal_init_dens_bias'], 'orange', marker='o', linestyle='-',
                  label='Initial Bias', linewidth=2, markersize=6)
    ax1_dens.plot(eval_points, metrics['hal_final_dens_bias'], 'go-', 
                  label='Final Bias', linewidth=2, markersize=6)
    ax1_dens.plot(eval_points, metrics['kde_dens_bias'], 'ro-', 
                  label='KDE Bias', linewidth=2, markersize=6)
    
    ax1_dens.set_xlabel('Evaluation Point')
    ax1_dens.set_ylabel('Bias')
    ax1_dens.set_title('Bias Across Evaluation Points')
    ax1_dens.grid(True)
    ax1_dens.legend()
    
    # Plot 2: Variance across evaluation points for density
    ax2_dens = fig_dens.add_subplot(gs_dens[0, 1])
    ax2_dens.plot(eval_points, metrics['hal_init_dens_variance'], 'orange', marker='o', linestyle='-',
                  label='Initial Variance', linewidth=2, markersize=6)
    ax2_dens.plot(eval_points, metrics['hal_final_dens_variance'], 'go-', 
                  label='Final Variance', linewidth=2, markersize=6)
    ax2_dens.plot(eval_points, metrics['kde_dens_variance'], 'ro-', 
                  label='KDE Variance', linewidth=2, markersize=6)
    
    ax2_dens.set_xlabel('Evaluation Point')
    ax2_dens.set_ylabel('Variance')
    ax2_dens.set_title('Variance Across Evaluation Points')
    ax2_dens.grid(True)
    ax2_dens.legend()
    
    # Plot 3: MSE across evaluation points for density
    ax3_dens = fig_dens.add_subplot(gs_dens[0, 2])
    ax3_dens.plot(eval_points, metrics['hal_init_dens_mse'], 'orange', marker='o', linestyle='-',
                  label='Initial MSE', linewidth=2, markersize=6)
    ax3_dens.plot(eval_points, metrics['hal_final_dens_mse'], 'go-', 
                  label='Final MSE', linewidth=2, markersize=6)
    ax3_dens.plot(eval_points, metrics['kde_dens_mse'], 'ro-', 
                  label='KDE MSE', linewidth=2, markersize=6)
    
    ax3_dens.set_xlabel('Evaluation Point')
    ax3_dens.set_ylabel('MSE')
    ax3_dens.set_title('MSE Across Evaluation Points')
    ax3_dens.grid(True)
    ax3_dens.legend()
    
    # Add overall title for density plots
    plt.suptitle(f'Density Function Metrics for Sample Size n={n_samples}', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for the suptitle
    
    # Save the figure for density
    fig_dens.savefig(f"{output_dir}/density_metrics_n{n_samples}.png", dpi=300, bbox_inches='tight')
    plt.close(fig_dens)
    
    return fig_surv, fig_dens

# Function to plot summarized metrics across all sample sizes

def plot_sample_size_summary(summary_data, output_dir):
    """
    Create side-by-side plots of mean metrics across sample sizes
    for both density and survival functions using normal scales.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Convert to DataFrame for easier plotting
    summary_df = pd.DataFrame(summary_data)
    
    # ----- SURVIVAL FUNCTION SUMMARY PLOTS -----
    # Create figure with 3 subplots side by side for survival
    fig_surv = plt.figure(figsize=(18, 6))
    gs_surv = GridSpec(1, 3, figure=fig_surv, wspace=0.3)
    
    # Plot 1: Mean Absolute Bias vs Sample Size (Survival) - NORMAL SCALE
    ax1_surv = fig_surv.add_subplot(gs_surv[0, 0])
    ax1_surv.plot(summary_df['sample_size'], summary_df['km_surv_mean_abs_bias'], 'bo-', 
                  label='Kaplan-Meier', linewidth=2, markersize=8)
    ax1_surv.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_abs_bias'], 'orange', marker='o', 
                  linestyle='-', label='HAL Initial', linewidth=2, markersize=8)
    ax1_surv.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_abs_bias'], 'go-', 
                  label='HAL Final', linewidth=2, markersize=8)
    ax1_surv.plot(summary_df['sample_size'], summary_df['kde_surv_mean_abs_bias'], 'ro-', 
                  label='KDE', linewidth=2, markersize=8)
    
    ax1_surv.set_xlabel('Sample Size')
    ax1_surv.set_ylabel('Mean Absolute Bias')
    ax1_surv.set_title('Mean Absolute Bias vs Sample Size')
    # Removed log scale transformations
    ax1_surv.grid(True)
    ax1_surv.legend()
    
    # Plot 2: Mean Variance vs Sample Size (Survival) - NORMAL SCALE
    ax2_surv = fig_surv.add_subplot(gs_surv[0, 1])
    ax2_surv.plot(summary_df['sample_size'], summary_df['km_surv_mean_variance'], 'bo-', 
                  label='Kaplan-Meier', linewidth=2, markersize=8)
    ax2_surv.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_variance'], 'orange', marker='o', 
                  linestyle='-', label='HAL Initial', linewidth=2, markersize=8)
    ax2_surv.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_variance'], 'go-', 
                  label='HAL Final', linewidth=2, markersize=8)
    ax2_surv.plot(summary_df['sample_size'], summary_df['kde_surv_mean_variance'], 'ro-', 
                  label='KDE', linewidth=2, markersize=8)
    
    ax2_surv.set_xlabel('Sample Size')
    ax2_surv.set_ylabel('Mean Variance')
    ax2_surv.set_title('Mean Variance vs Sample Size')
    # Removed log scale transformations
    ax2_surv.grid(True)
    ax2_surv.legend()
    
    # Plot 3: Mean MSE vs Sample Size (Survival) - NORMAL SCALE
    ax3_surv = fig_surv.add_subplot(gs_surv[0, 2])
    ax3_surv.plot(summary_df['sample_size'], summary_df['km_surv_mean_mse'], 'bo-', 
                  label='Kaplan-Meier', linewidth=2, markersize=8)
    ax3_surv.plot(summary_df['sample_size'], summary_df['hal_init_surv_mean_mse'], 'orange', marker='o', 
                  linestyle='-', label='HAL Initial', linewidth=2, markersize=8)
    ax3_surv.plot(summary_df['sample_size'], summary_df['hal_final_surv_mean_mse'], 'go-', 
                  label='HAL Final', linewidth=2, markersize=8)
    ax3_surv.plot(summary_df['sample_size'], summary_df['kde_surv_mean_mse'], 'ro-', 
                  label='KDE', linewidth=2, markersize=8)
    
    ax3_surv.set_xlabel('Sample Size')
    ax3_surv.set_ylabel('Mean MSE')
    ax3_surv.set_title('Mean MSE vs Sample Size')
    # Removed log scale transformations
    ax3_surv.grid(True)
    ax3_surv.legend()
    
    # Add overall title for survival
    plt.suptitle('Survival Function Metrics by Sample Size', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for the suptitle
    
    # Save the figure for survival
    fig_surv.savefig(f"{output_dir}/survival_metrics_by_sample_size_normal_scale.png", dpi=300, bbox_inches='tight')
    plt.close(fig_surv)
    
    # ----- DENSITY FUNCTION SUMMARY PLOTS -----
    # Create figure with 3 subplots side by side for density
    fig_dens = plt.figure(figsize=(18, 6))
    gs_dens = GridSpec(1, 3, figure=fig_dens, wspace=0.3)
    
    # Plot 1: Mean Absolute Bias vs Sample Size (Density) - NORMAL SCALE
    ax1_dens = fig_dens.add_subplot(gs_dens[0, 0])
    ax1_dens.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_abs_bias'], 'orange', marker='o', 
                  linestyle='-', label='HAL Initial', linewidth=2, markersize=8)
    ax1_dens.plot(summary_df['sample_size'], summary_df['hal_final_dens_mean_abs_bias'], 'go-', 
                  label='HAL Final', linewidth=2, markersize=8)
    ax1_dens.plot(summary_df['sample_size'], summary_df['kde_dens_mean_abs_bias'], 'ro-', 
                  label='KDE', linewidth=2, markersize=8)
    
    ax1_dens.set_xlabel('Sample Size')
    ax1_dens.set_ylabel('Mean Absolute Bias')
    ax1_dens.set_title('Mean Absolute Bias vs Sample Size')
    # Removed log scale transformations
    ax1_dens.grid(True)
    ax1_dens.legend()
    
    # Plot 2: Mean Variance vs Sample Size (Density) - NORMAL SCALE
    ax2_dens = fig_dens.add_subplot(gs_dens[0, 1])
    ax2_dens.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_variance'], 'orange', marker='o', 
                  linestyle='-', label='HAL Initial', linewidth=2, markersize=8)
    ax2_dens.plot(summary_df['sample_size'], summary_df['hal_final_dens_mean_variance'], 'go-', 
                  label='HAL Final', linewidth=2, markersize=8)
    ax2_dens.plot(summary_df['sample_size'], summary_df['kde_dens_mean_variance'], 'ro-', 
                  label='KDE', linewidth=2, markersize=8)
    
    ax2_dens.set_xlabel('Sample Size')
    ax2_dens.set_ylabel('Mean Variance')
    ax2_dens.set_title('Mean Variance vs Sample Size')
    # Removed log scale transformations
    ax2_dens.grid(True)
    ax2_dens.legend()
    
    # Plot 3: Mean MSE vs Sample Size (Density) - NORMAL SCALE
    ax3_dens = fig_dens.add_subplot(gs_dens[0, 2])
    ax3_dens.plot(summary_df['sample_size'], summary_df['hal_init_dens_mean_mse'], 'orange', marker='o', 
                  linestyle='-', label='HAL Initial', linewidth=2, markersize=8)
    ax3_dens.plot(summary_df['sample_size'], summary_df['hal_final_dens_mean_mse'], 'go-', 
                  label='HAL Final', linewidth=2, markersize=8)
    ax3_dens.plot(summary_df['sample_size'], summary_df['kde_dens_mean_mse'], 'ro-', 
                  label='KDE', linewidth=2, markersize=8)
    
    ax3_dens.set_xlabel('Sample Size')
    ax3_dens.set_ylabel('Mean MSE')
    ax3_dens.set_title('Mean MSE vs Sample Size')
    # Removed log scale transformations
    ax3_dens.grid(True)
    ax3_dens.legend()
    
    # Add overall title for density
    plt.suptitle('Density Function Metrics by Sample Size', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for the suptitle
    
    # Save the figure for density - Note the different filename
    fig_dens.savefig(f"{output_dir}/density_metrics_by_sample_size_normal_scale.png", dpi=300, bbox_inches='tight')
    plt.close(fig_dens)
    
    return fig_surv, fig_dens
# Main execution loop
for n_samples in sample_sizes:
    print(f"\n{'='*70}")
    print(f"ANALYZING SAMPLE SIZE n = {n_samples}")
    print(f"{'='*70}\n")
    
    # Set sample-specific directories
    out_comparison_dir = f"{comparison_base_dir}/out_comparison_{n_samples}"
    metrics_dir = f"{comparison_base_dir}/metrics_{n_samples}"
    
    # Create output directories
    os.makedirs(out_comparison_dir, exist_ok=True)
    os.makedirs(metrics_dir, exist_ok=True)
    
    # Step 1: Generate data if needed
    if not os.path.exists(f"{out_comparison_dir}/comparison_n{n_samples}_kde_density.csv"):
        generate_comparison_data(n_samples, n_generate, evaluation_points, out_comparison_dir)
    else:
        print(f"Using existing comparison data from {out_comparison_dir}")
    
    # Step 2: Load data
    try:
        print(f"Loading HAL results from {hal_dir} and comparison results from {out_comparison_dir}...")
        hal_data, comparison_data = load_data_fixed(
            hal_dir, out_comparison_dir, n_samples, smoothness_order, ridge
        )
        
        # Step 3: Compute performance metrics
        print("Computing performance metrics across evaluation points...")
        metrics = compute_performance_metrics(hal_data, comparison_data)
        
        if metrics is not None:
            # Step 4: Create side-by-side plots of bias, variance, and MSE
            print("Creating side-by-side plots of bias, variance, and MSE across evaluation points...")
            plot_metrics_by_evaluation_point(metrics, metrics_dir, n_samples)
            
            # Step 5: Compute and store mean metrics for cross-sample size analysis
            print("Computing mean metrics for cross-sample size analysis...")
            sample_size_summary['sample_size'].append(n_samples)
            
            # Survival function metrics
            # Mean absolute bias
            sample_size_summary['hal_init_surv_mean_abs_bias'].append(np.mean(np.abs(metrics['hal_init_surv_bias'])))
            sample_size_summary['hal_final_surv_mean_abs_bias'].append(np.mean(np.abs(metrics['hal_final_surv_bias'])))
            sample_size_summary['kde_surv_mean_abs_bias'].append(np.mean(np.abs(metrics['kde_surv_bias'])))
            sample_size_summary['km_surv_mean_abs_bias'].append(np.mean(np.abs(metrics['km_surv_bias'])))
            
            # Mean variance
            sample_size_summary['hal_init_surv_mean_variance'].append(np.mean(metrics['hal_init_surv_variance']))
            sample_size_summary['hal_final_surv_mean_variance'].append(np.mean(metrics['hal_final_surv_variance']))
            sample_size_summary['kde_surv_mean_variance'].append(np.mean(metrics['kde_surv_variance']))
            sample_size_summary['km_surv_mean_variance'].append(np.mean(metrics['km_surv_variance']))
            
            # Mean MSE
            sample_size_summary['hal_init_surv_mean_mse'].append(np.mean(metrics['hal_init_surv_mse']))
            sample_size_summary['hal_final_surv_mean_mse'].append(np.mean(metrics['hal_final_surv_mse']))
            sample_size_summary['kde_surv_mean_mse'].append(np.mean(metrics['kde_surv_mse']))
            sample_size_summary['km_surv_mean_mse'].append(np.mean(metrics['km_surv_mse']))
            
            # Density function metrics
            # Mean absolute bias
            sample_size_summary['hal_init_dens_mean_abs_bias'].append(np.mean(np.abs(metrics['hal_init_dens_bias'])))
            sample_size_summary['hal_final_dens_mean_abs_bias'].append(np.mean(np.abs(metrics['hal_final_dens_bias'])))
            sample_size_summary['kde_dens_mean_abs_bias'].append(np.mean(np.abs(metrics['kde_dens_bias'])))
            
            # Mean variance
            sample_size_summary['hal_init_dens_mean_variance'].append(np.mean(metrics['hal_init_dens_variance']))
            sample_size_summary['hal_final_dens_mean_variance'].append(np.mean(metrics['hal_final_dens_variance']))
            sample_size_summary['kde_dens_mean_variance'].append(np.mean(metrics['kde_dens_variance']))
            
            # Mean MSE
            sample_size_summary['hal_init_dens_mean_mse'].append(np.mean(metrics['hal_init_dens_mse']))
            sample_size_summary['hal_final_dens_mean_mse'].append(np.mean(metrics['hal_final_dens_mse']))
            sample_size_summary['kde_dens_mean_mse'].append(np.mean(metrics['kde_dens_mse']))
            
            # Save metrics to CSV
            metrics_df = pd.DataFrame({
                'evaluation_point': metrics['eval_points'],
                # Survival metrics
                'hal_init_surv_bias': metrics['hal_init_surv_bias'],
                'hal_final_surv_bias': metrics['hal_final_surv_bias'],
                'kde_surv_bias': metrics['kde_surv_bias'],
                'km_surv_bias': metrics['km_surv_bias'],
                'hal_init_surv_variance': metrics['hal_init_surv_variance'],
                'hal_final_surv_variance': metrics['hal_final_surv_variance'],
                'kde_surv_variance': metrics['kde_surv_variance'],
                'km_surv_variance': metrics['km_surv_variance'],
                'hal_init_surv_mse': metrics['hal_init_surv_mse'],
                'hal_final_surv_mse': metrics['hal_final_surv_mse'],
                'kde_surv_mse': metrics['kde_surv_mse'],
                'km_surv_mse': metrics['km_surv_mse'],
                # Density metrics
                'hal_init_dens_bias': metrics['hal_init_dens_bias'],
                'hal_final_dens_bias': metrics['hal_final_dens_bias'],
                'kde_dens_bias': metrics['kde_dens_bias'],
                'hal_init_dens_variance': metrics['hal_init_dens_variance'],
                'hal_final_dens_variance': metrics['hal_final_dens_variance'],
                'kde_dens_variance': metrics['kde_dens_variance'],
                'hal_init_dens_mse': metrics['hal_init_dens_mse'],
                'hal_final_dens_mse': metrics['hal_final_dens_mse'],
                'kde_dens_mse': metrics['kde_dens_mse']
            })
            
            metrics_df.to_csv(f"{metrics_dir}/metrics_n{n_samples}.csv", index=False)
            
            print(f"Analysis complete for sample size n={n_samples}.")
        else:
            print(f"Could not compute metrics for sample size n={n_samples} due to lack of common experiments.")
    except Exception as e:
        print(f"Error processing sample size n={n_samples}: {str(e)}")
        print("Skipping to next sample size...")
        continue

# Create a sample size summary analysis if we have data for at least 2 sample sizes
if len(sample_size_summary['sample_size']) >= 2:
    print("\n\n" + "="*80)
    print("CROSS-SAMPLE SIZE COMPARISON ANALYSIS")
    print("="*80 + "\n")
    
    # Create and save the summary plots
    summary_dir = f"{comparison_base_dir}/sample_size_summary"
    os.makedirs(summary_dir, exist_ok=True)
    
    # Create the summary plots for both density and survival function metrics
    plot_sample_size_summary(sample_size_summary, summary_dir)
    
    # Save summary metrics to CSV
    summary_df = pd.DataFrame(sample_size_summary)
    summary_df.to_csv(f"{summary_dir}/metrics_by_sample_size_summary.csv", index=False)
    
    print("\nConvergence Rate Estimates:")
    
    # Compute convergence rates for survival function metrics
    log_n = np.log(summary_df['sample_size'])
    
    print("\n----- SURVIVAL FUNCTION CONVERGENCE RATES -----")
    
    # HAL final bias (survival)
    log_hal_final_surv_bias = np.log(summary_df['hal_final_surv_mean_abs_bias'])
    hal_final_surv_bias_slope, _, _, _, _ = linregress(log_n, log_hal_final_surv_bias)
    print(f"HAL Final Survival Bias Convergence Rate: approximately n^({hal_final_surv_bias_slope:.2f})")
    
    # KM bias (survival)
    log_km_surv_bias = np.log(summary_df['km_surv_mean_abs_bias'])
    km_surv_bias_slope, _, _, _, _ = linregress(log_n, log_km_surv_bias)
    print(f"Kaplan-Meier Bias Convergence Rate: approximately n^({km_surv_bias_slope:.2f})")
    
    # HAL final variance (survival)
    log_hal_final_surv_var = np.log(summary_df['hal_final_surv_mean_variance'])
    hal_final_surv_var_slope, _, _, _, _ = linregress(log_n, log_hal_final_surv_var)
    print(f"HAL Final Survival Variance Convergence Rate: approximately n^({hal_final_surv_var_slope:.2f})")
    
    # KM variance (survival)
    log_km_surv_var = np.log(summary_df['km_surv_mean_variance'])
    km_surv_var_slope, _, _, _, _ = linregress(log_n, log_km_surv_var)
    print(f"Kaplan-Meier Variance Convergence Rate: approximately n^({km_surv_var_slope:.2f})")
    
    # HAL final MSE (survival)
    log_hal_final_surv_mse = np.log(summary_df['hal_final_surv_mean_mse'])
    hal_final_surv_mse_slope, _, _, _, _ = linregress(log_n, log_hal_final_surv_mse)
    print(f"HAL Final Survival MSE Convergence Rate: approximately n^({hal_final_surv_mse_slope:.2f})")
    
    # KM MSE (survival)
    log_km_surv_mse = np.log(summary_df['km_surv_mean_mse'])
    km_surv_mse_slope, _, _, _, _ = linregress(log_n, log_km_surv_mse)
    print(f"Kaplan-Meier MSE Convergence Rate: approximately n^({km_surv_mse_slope:.2f})")
    
    print("\n----- DENSITY FUNCTION CONVERGENCE RATES -----")
    
    # HAL final bias (density)
    log_hal_final_dens_bias = np.log(summary_df['hal_final_dens_mean_abs_bias'])
    hal_final_dens_bias_slope, _, _, _, _ = linregress(log_n, log_hal_final_dens_bias)
    print(f"HAL Final Density Bias Convergence Rate: approximately n^({hal_final_dens_bias_slope:.2f})")
    
    # KDE bias (density)
    log_kde_dens_bias = np.log(summary_df['kde_dens_mean_abs_bias'])
    kde_dens_bias_slope, _, _, _, _ = linregress(log_n, log_kde_dens_bias)
    print(f"KDE Bias Convergence Rate: approximately n^({kde_dens_bias_slope:.2f})")
    
    # HAL final variance (density)
    log_hal_final_dens_var = np.log(summary_df['hal_final_dens_mean_variance'])
    hal_final_dens_var_slope, _, _, _, _ = linregress(log_n, log_hal_final_dens_var)
    print(f"HAL Final Density Variance Convergence Rate: approximately n^({hal_final_dens_var_slope:.2f})")
    
    # KDE variance (density)
    log_kde_dens_var = np.log(summary_df['kde_dens_mean_variance'])
    kde_dens_var_slope, _, _, _, _ = linregress(log_n, log_kde_dens_var)
    print(f"KDE Variance Convergence Rate: approximately n^({kde_dens_var_slope:.2f})")
    
    # HAL final MSE (density)
    log_hal_final_dens_mse = np.log(summary_df['hal_final_dens_mean_mse'])
    hal_final_dens_mse_slope, _, _, _, _ = linregress(log_n, log_hal_final_dens_mse)
    print(f"HAL Final Density MSE Convergence Rate: approximately n^({hal_final_dens_mse_slope:.2f})")
    
    # KDE MSE (density)
    log_kde_dens_mse = np.log(summary_df['kde_dens_mean_mse'])
    kde_dens_mse_slope, _, _, _, _ = linregress(log_n, log_kde_dens_mse)
    print(f"KDE MSE Convergence Rate: approximately n^({kde_dens_mse_slope:.2f})")
    
    # Comparison of convergence rates
    print("\n----- CONVERGENCE RATE COMPARISON -----")
    
    if hal_final_surv_mse_slope < km_surv_mse_slope:
        print(f"HAL Targeted Survival shows faster MSE convergence than Kaplan-Meier ({hal_final_surv_mse_slope:.2f} vs {km_surv_mse_slope:.2f})")
    else:
        print(f"Kaplan-Meier shows faster MSE convergence than HAL Targeted Survival ({km_surv_mse_slope:.2f} vs {hal_final_surv_mse_slope:.2f})")
        
    if hal_final_dens_mse_slope < kde_dens_mse_slope:
        print(f"HAL Targeted Density shows faster MSE convergence than KDE ({hal_final_dens_mse_slope:.2f} vs {kde_dens_mse_slope:.2f})")
    else:
        print(f"KDE shows faster MSE convergence than HAL Targeted Density ({kde_dens_mse_slope:.2f} vs {hal_final_dens_mse_slope:.2f})")
    
    print("\nAnalysis complete! Check the metrics directories and sample_size_summary directory for all plots and results.")
else:
    print("\nNot enough sample sizes analyzed to create cross-sample size comparison.")
    print("Please run the analysis on at least 2 different sample sizes.")
    


ANALYZING SAMPLE SIZE n = 200

Using existing comparison data from ./out_comparison_200
Loading HAL results from ../Asymptoticity_FD_0301/out and comparison results from ./out_comparison_200...
Computing performance metrics across evaluation points...
Computing metrics using 500 common experiments.
Creating side-by-side plots of bias, variance, and MSE across evaluation points...


/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_6748/3507623215.py:425: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for the suptitle
/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_6748/3507623215.py:483: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for the suptitle


Computing mean metrics for cross-sample size analysis...
Analysis complete for sample size n=200.

ANALYZING SAMPLE SIZE n = 400

Using existing comparison data from ./out_comparison_400
Loading HAL results from ../Asymptoticity_FD_0301/out and comparison results from ./out_comparison_400...
Computing performance metrics across evaluation points...
Computing metrics using 500 common experiments.
Creating side-by-side plots of bias, variance, and MSE across evaluation points...


/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_6748/3507623215.py:425: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for the suptitle
/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_6748/3507623215.py:483: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for the suptitle


Computing mean metrics for cross-sample size analysis...
Analysis complete for sample size n=400.

ANALYZING SAMPLE SIZE n = 800

Using existing comparison data from ./out_comparison_800
Loading HAL results from ../Asymptoticity_FD_0301/out and comparison results from ./out_comparison_800...
Computing performance metrics across evaluation points...
Computing metrics using 500 common experiments.
Creating side-by-side plots of bias, variance, and MSE across evaluation points...


...

Computing mean metrics for cross-sample size analysis...
Analysis complete for sample size n=1600.

ANALYZING SAMPLE SIZE n = 3200

Using existing comparison data from ./out_comparison_3200
Loading HAL results from ../Asymptoticity_FD_0301/out and comparison results from ./out_comparison_3200...
Computing performance metrics across evaluation points...
Computing metrics using 500 common experiments.
Creating side-by-side plots of bias, variance, and MSE across evaluation points...


/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_6748/3507623215.py:425: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for the suptitle
/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_6748/3507623215.py:483: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for the suptitle


Computing mean metrics for cross-sample size analysis...
Analysis complete for sample size n=3200.


CROSS-SAMPLE SIZE COMPARISON ANALYSIS



/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_6748/3507623215.py:564: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for the suptitle
/var/folders/65/x30mj4jd3wb_rz8hlyy732_m0000gn/T/ipykernel_6748/3507623215.py:625: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust for the suptitle



Convergence Rate Estimates:

----- SURVIVAL FUNCTION CONVERGENCE RATES -----
HAL Final Survival Bias Convergence Rate: approximately n^(0.03)
Kaplan-Meier Bias Convergence Rate: approximately n^(-0.53)
HAL Final Survival Variance Convergence Rate: approximately n^(-0.92)
Kaplan-Meier Variance Convergence Rate: approximately n^(-0.97)
HAL Final Survival MSE Convergence Rate: approximately n^(-0.81)
Kaplan-Meier MSE Convergence Rate: approximately n^(-0.97)

----- DENSITY FUNCTION CONVERGENCE RATES -----
HAL Final Density Bias Convergence Rate: approximately n^(0.01)
KDE Bias Convergence Rate: approximately n^(-0.39)
HAL Final Density Variance Convergence Rate: approximately n^(-0.83)
KDE Variance Convergence Rate: approximately n^(-0.72)
HAL Final Density MSE Convergence Rate: approximately n^(-0.74)
KDE MSE Convergence Rate: approximately n^(-0.73)

----- CONVERGENCE RATE COMPARISON -----
Kaplan-Meier shows faster MSE convergence than HAL Targeted Survival (-0.97 vs -0.81)
HAL Targete